# 04 — vLLM: Needle-in-a-Haystack Baseline

This notebook runs the same NIAH benchmark as notebook 02, but using
[vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B **without
KV cache compression**.

vLLM uses PagedAttention for efficient memory management. This serves
as the quality and throughput baseline for comparison against
KeyDiffPress in notebook 05.

Results are saved to `results/vllm/` for comparison in notebook 05.

## 1 Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

NEEDLE_DEPTHS = [0, 25, 50, 75, 100]

MAX_CONTEXT_LENGTHS = [4096, 8192]

MAX_NEW_TOKENS = 64

PRESS_CONFIGS = {
    "full_replacement": lambda cr: 
    {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "max_model_len": max(MAX_CONTEXT_LENGTHS) + MAX_NEW_TOKENS + 256,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "max_model_len": max(MAX_CONTEXT_LENGTHS) + MAX_NEW_TOKENS + 256,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [ ]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n⚠  GPU memory is not free — a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()    

## 2. Load Haystack Dataset

Paul Graham essays used as haystack filler, following the original
Needle-in-a-Haystack benchmark by [Kamradt (2023)](https://github.com/gkamradt/LLMTest_NeedleInAHaystack).

In [ ]:
from datasets import load_dataset

haystack_ds = load_dataset("alessiodevoto/paul_graham_essays", split="test")
haystack_df = haystack_ds.to_pandas()

print(f"Haystack dataset loaded: {len(haystack_df)} rows")
print(f"Columns: {list(haystack_df.columns)}")
print(f"Needle: {haystack_df['needle'].iloc[0][:100]}...")
print(f"Question: {haystack_df['question'].iloc[0]}")

## 3. Prepare Prompts

For each (max_context_length, needle_depth) combination, we build the
haystack with needle inserted, then apply the model's chat template
to produce the final prompt for vLLM.

In [ ]:
from transformers import AutoTokenizer
from eval_utils import insert_needle_in_haystack

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompt_configs = []

for max_ctx_len in MAX_CONTEXT_LENGTHS:
    niah_df = insert_needle_in_haystack(
        haystack_df, tokenizer, max_ctx_len, NEEDLE_DEPTHS,
    )

    for idx, row in niah_df.iterrows():
        user_msg = row["context"] + "\n\n" + row["question"]
        messages = [{"role": "user", "content": user_msg}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        if row["answer_prefix"]:
            prompt += row["answer_prefix"]

        prompt_configs.append({
            "prompt": prompt,
            "max_context_length": max_ctx_len,
            "needle_depth": row["needle_depth"],
            "needle": row["needle"],
            "question": row["question"],
        })

print(f"Prepared {len(prompt_configs)} prompts")

## 4. Run Batch Inference

vLLM processes all prompts in a single batch via its internal scheduler.

In [ ]:
import time
import json
from eval_utils import calculate_niah_metrics, rouge_l_f_scores
from vllm import LLM, SamplingParams

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

def get_gpu_memory_used_gb() -> float:
    """Actual GPU memory used, measured at the CUDA driver level.
    Works for vLLM which manages its own memory pool outside PyTorch."""
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

prompts = [pc["prompt"] for pc in prompt_configs]

all_results = []
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        try:
            press = press_factory(ratio)
            llm = LLM(**press)
    
            mem_before = get_gpu_memory_used_gb()
            start = time.perf_counter()
    
            outputs = llm.generate(prompts, sampling_params)
    
            batch_elapsed = time.perf_counter() - start
            mem_after = get_gpu_memory_used_gb()
            peak_mem = max(mem_before, mem_after)
    
            for i, output in enumerate(outputs):
                predicted_answer = output.outputs[0].text.strip()
                pc = prompt_configs[i]
    
                result = {
                    "framework": "vllm",
                    "press": press_name,
                    "compression_ratio": ratio,
                    "max_context_length": pc["max_context_length"],
                    "needle_depth": pc["needle_depth"],
                    "needle": pc["needle"],
                    "question": pc["question"],
                    "predicted_answer": predicted_answer,
                    "elapsed_sec": round(batch_elapsed / len(prompts), 3),
                    "peak_gpu_mem_gb": round(peak_mem, 3),
                    "task": "needle_in_haystack",
                }
                all_results.append(result)
    
            total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
            throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0
    
            print(f"Batch done in {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")
        finally:
            cleanup_vllm(llm)
print(f"Total results: {len(all_results)}")

## 5. Score Predictions

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

metrics = calculate_niah_metrics(df)
df["rouge_l_f"] = rouge_l_f_scores(metrics)

summary = (
    df.groupby(["press", "compression_ratio", "max_context_length", "needle_depth"])
    .agg(
        rouge_l=("rouge_l_f", "mean"),
        mean_time=("elapsed_sec", "mean"),
        peak_mem=("peak_gpu_mem_gb", "mean"),
    )
    .round(4)
)

print(summary.to_string())

## 6. Save Results

In [ ]:
import os

os.makedirs("results/vllm", exist_ok=True)

predictions_path = "results/vllm/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")